# Multi-Physics Climate Modeling

[Full course sequence](../../ai4sci/README.md) · [Start notebook](../../Start_Here.ipynb)

All levels are retained and use PhysicsNeMo **2.2.2** `FullyConnected`, SymPy `PDE`, `PhysicsInformer`, and explicit PyTorch training loops. Complete `student_equations` in the linked `.py` file. Until that function is completed, the default run stops with an explanatory error. If you get stuck, set `USE_REFERENCE=True` to run and compare the completed implementation.

**Learning workflow:** inspect the equations and conditions → edit and save the linked `.py` file → run it → inspect the PDE and condition errors and the predictions. A successful short run does not establish convergence. Each run writes to a new directory.

`PhysicsInformer` computes spatial derivatives from `coordinates`. For the current API, the training code computes time derivatives with PyTorch autograd and supplies keys such as `u__t` and `u__t__t`. Read `loss_terms` and the explicit `optimizer.zero_grad → backward → step` loop in each training file.

Instructor automation can execute these same cells with the environment variables `AI4SCI_REFERENCE=1`, `AI4SCI_DEVICE=cpu`, and `AI4SCI_STEPS=2`. The default remains the student exercise mode.


In [ ]:
from pathlib import Path
from datetime import datetime
import json
import os
import subprocess
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "challenge" / "climate" / "climate_l1.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open the notebook from inside the repository.")
LAB_DIR = ROOT / "challenge" / "climate"
USE_REFERENCE = os.environ.get("AI4SCI_REFERENCE", "0").lower() in {"1", "true", "yes"}  # Default: student exercise mode.
DEVICE = os.environ.get("AI4SCI_DEVICE", "auto")
STEPS = int(os.environ.get("AI4SCI_STEPS", "200"))  # Verify runtime and accuracy on the event GPU.
SEED = 42
RUN_TAG = datetime.now().strftime("%Y%m%d-%H%M%S-%f")


## Level 1 · Simple Atmosphere Model

This temperature advection–diffusion–reaction problem uses the spatial domain $[0,\pi]^2$ and time interval $[0,2\pi]$.
$$T_t+u_0T_x+v_0T_y-\kappa\Delta T-Q_0+\lambda(T-T_{eq})=0.$$
The initial value is $\sin x\sin y$, and the temperature is zero on all four edges. The defaults $u_0=v_0=Q_0=\lambda=0,\kappa=1$ reduce the equation to diffusion.
Only this default verification case uses $T=\sin x\sin y\,e^{-2\kappa t}$. When you change physical parameters in `conf/config_atmos.yaml`, the code checks whether the assumptions needed for this analytic reference still hold. This example does not run the FourCastNet weather-forecasting model.

### Code and exercise

Open [climate_l1.py](climate_l1.py) and inspect `reference_equations`, `student_equations`, `loss_terms`, and `main`. Write the required dictionary of PDE residuals in `student_equations`, then save with **Ctrl+S / ⌘S**. Identify where the applicable initial, boundary, and integral conditions enter the loss.

Start with a small run equivalent to `--steps 2 --device cpu --reference` to check the complete input/output path. The student and reference implementations share the same sampling and evaluation code.


In [ ]:
RUN_COMPLETED = False
result_dir = LAB_DIR / "outputs" / f"climate_l1-{RUN_TAG}"
command = [sys.executable, str(LAB_DIR / "climate_l1.py"),
           "--steps", str(STEPS), "--seed", str(SEED), "--device", DEVICE,
           "--output-dir", str(result_dir)]
if USE_REFERENCE:
    command.append("--reference")
subprocess.run(command, cwd=LAB_DIR, check=True)
RUN_COMPLETED = True


### Inspect the actual results

The first and last held-out rows in `loss.csv` are evaluated at **identical points**. Intermediate rows describe freshly sampled training minibatches. Distinguish the initial/final errors from the PDE and condition residuals in `metrics.json`. A reference error is included only when a comparable reference is available. `model.pt` stores the model state and configuration; `predictions.npz` contains the actual prediction arrays.


In [ ]:
if not RUN_COMPLETED:
    raise RuntimeError("The current training run has not completed.")
metrics = json.loads((result_dir / "metrics.json").read_text())
assert metrics["steps"] == STEPS and metrics["seed"] == SEED
print(json.dumps(metrics, indent=2))
preview = result_dir / "preview.png"
if preview.is_file():
    from IPython.display import display, Image
    display(Image(filename=str(preview)))
else:
    print("Plotting dependencies are unavailable. Inspect the actual arrays in predictions.npz.")


## Level 2 · Coupled Atmosphere–Ocean System

Predict the atmospheric temperature $T_a$ and ocean temperature $T_o$ together on the same space-time domain.
$$T_{a,t}+u_0T_{a,x}+v_0T_{a,y}-\kappa_a\Delta T_a-Q_a+\lambda_a(T_a-T_{eq,a})+\gamma(T_a-T_o)=0,$$
$$T_{o,t}-\kappa_o\Delta T_o-Q_o-\gamma(T_a-T_o)=0.$$
The exchange terms have opposite signs, so internal heat exchange cancels when the two equations are added. Both initial values are $\sin x\sin y$, and both boundary values are zero.
The default verification case is the uncoupled special case $\gamma=0$, with $\kappa_a=1,\kappa_o=.5$. Compare each field with its sine-decay reference only when these assumptions apply. Set `gamma0` to a positive value in `conf/config_coupled.yaml` to train the coupled system; the uncoupled reference is then disabled automatically. Check the other parameters against the reference assumptions as well.

### Code and exercise

Open [climate_l2.py](climate_l2.py) and inspect `reference_equations`, `student_equations`, `loss_terms`, and `main`. Write the required dictionary of PDE residuals in `student_equations`, then save with **Ctrl+S / ⌘S**. Identify where the applicable initial, boundary, and integral conditions enter the loss.

Start with a small run equivalent to `--steps 2 --device cpu --reference` to check the complete input/output path. The student and reference implementations share the same sampling and evaluation code.


In [ ]:
RUN_COMPLETED = False
result_dir = LAB_DIR / "outputs" / f"climate_l2-{RUN_TAG}"
command = [sys.executable, str(LAB_DIR / "climate_l2.py"),
           "--steps", str(STEPS), "--seed", str(SEED), "--device", DEVICE,
           "--output-dir", str(result_dir)]
if USE_REFERENCE:
    command.append("--reference")
subprocess.run(command, cwd=LAB_DIR, check=True)
RUN_COMPLETED = True


### Inspect the actual results

The first and last held-out rows in `loss.csv` are evaluated at **identical points**. Intermediate rows describe freshly sampled training minibatches. Distinguish the initial/final errors from the PDE and condition residuals in `metrics.json`. A reference error is included only when a comparable reference is available. `model.pt` stores the model state and configuration; `predictions.npz` contains the actual prediction arrays.


In [ ]:
if not RUN_COMPLETED:
    raise RuntimeError("The current training run has not completed.")
metrics = json.loads((result_dir / "metrics.json").read_text())
assert metrics["steps"] == STEPS and metrics["seed"] == SEED
print(json.dumps(metrics, indent=2))
preview = result_dir / "preview.png"
if preview.is_file():
    from IPython.display import display, Image
    display(Image(filename=str(preview)))
else:
    print("Plotting dependencies are unavailable. Inspect the actual arrays in predictions.npz.")


## Check your understanding and continue

- How do the inputs, outputs, equations, and initial/boundary conditions change between levels?
- Are training minibatch loss and error at fixed validation points the same metric?
- Does the problem have an independent reference? If so, do its assumptions match the current configuration?
- Compare changes to sample counts, training steps, and condition weights using new run directories and saved configurations.

[Next challenge](../neural_operator/Advanced_Neural_Operators.ipynb) · [Full course sequence](../../ai4sci/README.md)

Adapted from the original OpenHackathons materials, with the file-specific copyright notices retained. [License](../../LICENSE).


--- 

Don't forget to check out additional [Open Hackathons Resources](https://www.openhackathons.org/s/technical-resources) and join our [OpenACC and Hackathons Slack Channel](https://www.openacc.org/community#slack) to share your experience and get more help from the community.

---

# Licensing

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials may include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.
